# Sprint 4 Multi-label — Ensemble v1 + v2

Promedia las predicciones de ambos modelos para aprovechar
las fortalezas de cada uno sin reentrenar.

**v1** mejor en: Hernia, Cardiomegaly, Pneumothorax
**v2** mejor en: Fibrosis, Pneumonia, Pleural_Thickening, Atelectasis

**Tiempo: ~20 min. Runtime: GPU o CPU.**


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR     = f'{PROJECT_ROOT}/code'
NIH_DIR      = f'{PROJECT_ROOT}/data/raw/nih'

import os, sys, shutil, time
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
import numpy as np

!pip install -q torchxrayvision h5py
print(f"Device: {'GPU' if torch.cuda.is_available() else 'CPU'}")

H5_LOCAL     = '/content/nih_images.h5'
PROCESSED_ML = '/content/processed_s4ml'
CLASS_NAMES_14 = [
    'Atelectasis','Cardiomegaly','Consolidation','Edema',
    'Effusion','Emphysema','Fibrosis','Hernia',
    'Infiltration','Mass','Nodule','Pleural_Thickening',
    'Pneumonia','Pneumothorax',
]

WANG_AUC = {
    'Atelectasis':0.7003,'Cardiomegaly':0.8100,'Consolidation':0.7032,
    'Edema':0.8052,'Effusion':0.7585,'Emphysema':0.8330,'Fibrosis':0.7859,
    'Hernia':0.8717,'Infiltration':0.6614,'Mass':0.6931,'Nodule':0.6693,
    'Pleural_Thickening':0.6835,'Pneumonia':0.6580,'Pneumothorax':0.7993,
}


## 2. Copiar HDF5 y CSVs

In [ ]:
import h5py

if not (os.path.exists(H5_LOCAL) and os.path.getsize(H5_LOCAL) > 4_000_000_000):
    print('Copiando HDF5...')
    shutil.copy2(f'{NIH_DIR}/nih_images.h5', H5_LOCAL)

with h5py.File(H5_LOCAL, 'r') as hf:
    print(f'✓ HDF5: {len(hf["images"]):,} imgs')

import pandas as pd
os.makedirs(PROCESSED_ML, exist_ok=True)
for csv in ['val.csv', 'test.csv']:
    local = f'{PROCESSED_ML}/{csv}'
    if not os.path.exists(local):
        shutil.copy(f'{PROJECT_ROOT}/data/processed_s4ml/{csv}', local)
    print(f'✓ {csv}: {len(pd.read_csv(local))} filas')


## 3. Cargar ambos modelos

In [ ]:
for mod in list(sys.modules.keys()):
    if mod.startswith('src.'): del sys.modules[mod]

from src.models.cnn_vit import CNNViT
from src.training.utils import get_device

device = get_device()

# Modelo v1: 4 capas ViT
model_v1 = CNNViT(
    backbone_weights='densenet121-res224-nih',
    num_classes=14, embedding_dim=512,
    num_heads=8, num_layers=4, mlp_dim=1024,
).to(device)
ckpt_v1 = torch.load(
    f'{PROJECT_ROOT}/experiments_s4ml/checkpoints/sprint4ml_phase2_best.pt',
    map_location=device
)
model_v1.load_state_dict(ckpt_v1['model_state_dict'])
model_v1.eval()
print(f'✓ v1 cargado (4 capas) | val AUC={ckpt_v1["best_val_auc_macro"]:.4f}')

# Modelo v2: 6 capas ViT
model_v2 = CNNViT(
    backbone_weights='densenet121-res224-nih',
    num_classes=14, embedding_dim=512,
    num_heads=8, num_layers=6, mlp_dim=1024,
).to(device)
ckpt_v2 = torch.load(
    f'{PROJECT_ROOT}/experiments_s4ml_v2/checkpoints/phase2_best.pt',
    map_location=device
)
model_v2.load_state_dict(ckpt_v2['model_state_dict'])
model_v2.eval()
print(f'✓ v2 cargado (6 capas) | val AUC={ckpt_v2["best_val_auc_macro"]:.4f}')


## 4. Obtener predicciones en val y test

In [ ]:
from torch.utils.data import DataLoader
from src.datasets.nih_multilabel import NIHDatasetMultiLabel
from src.datasets.transforms import build_eval_transform

eval_tfm = build_eval_transform(224)

def get_probs(model, csv_path):
    ds     = NIHDatasetMultiLabel(csv_path, H5_LOCAL, eval_tfm)
    loader = DataLoader(ds, batch_size=32, shuffle=False, num_workers=0)
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['image'].to(device))
            probs  = torch.sigmoid(logits.float())
            all_probs.append(probs.cpu())
            all_labels.append(batch['label'])
    return torch.cat(all_probs).numpy(), torch.cat(all_labels).numpy()

print('Obteniendo predicciones en val...')
val_probs_v1, val_labels = get_probs(model_v1, f'{PROCESSED_ML}/val.csv')
val_probs_v2, _          = get_probs(model_v2, f'{PROCESSED_ML}/val.csv')
print(f'✓ val: {len(val_labels)} muestras')

print('Obteniendo predicciones en test...')
test_probs_v1, test_labels = get_probs(model_v1, f'{PROCESSED_ML}/test.csv')
test_probs_v2, _           = get_probs(model_v2, f'{PROCESSED_ML}/test.csv')
print(f'✓ test: {len(test_labels)} muestras')


## 5. Evaluar pesos del ensemble en val

In [ ]:
from src.evaluation.metrics_multilabel import compute_metrics_multilabel

# Probar diferentes pesos para encontrar el óptimo en val
results = {}
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    probs = w * val_probs_v1 + (1-w) * val_probs_v2
    m     = compute_metrics_multilabel(val_labels, probs)
    results[w] = m['auc_macro']
    print(f'w_v1={w:.1f} w_v2={1-w:.1f}: AUC macro={m["auc_macro"]:.4f}')

# Agregar resultados individuales para comparar
m_v1 = compute_metrics_multilabel(val_labels, val_probs_v1)
m_v2 = compute_metrics_multilabel(val_labels, val_probs_v2)
print(f'\nv1 solo: AUC macro={m_v1["auc_macro"]:.4f}')
print(f'v2 solo: AUC macro={m_v2["auc_macro"]:.4f}')

best_w = max(results, key=results.get)
print(f'\n✓ Mejor peso: w_v1={best_w:.1f} → AUC macro={results[best_w]:.4f}')


## 6. Evaluar ensemble en TEST con el mejor peso

In [ ]:
# Ensemble con peso óptimo de val
probs_ensemble = best_w * test_probs_v1 + (1-best_w) * test_probs_v2

m_v1_test  = compute_metrics_multilabel(test_labels, test_probs_v1)
m_v2_test  = compute_metrics_multilabel(test_labels, test_probs_v2)
m_ens_test = compute_metrics_multilabel(test_labels, probs_ensemble)

print('RESULTADOS FINALES EN TEST:')
print('='*68)
print(f"{'Clase':<22} {'Wang17':>7} {'v1':>7} {'v2':>7} {'Ensemble':>9} {'Δ':>7}")
print('-'*68)

for i, cls in enumerate(CLASS_NAMES_14):
    wang = WANG_AUC.get(cls, 0)
    v1   = m_v1_test['auc_per_class'][i]
    v2   = m_v2_test['auc_per_class'][i]
    ens  = m_ens_test['auc_per_class'][i]
    if np.isnan(v1): v1 = 0
    if np.isnan(v2): v2 = 0
    if np.isnan(ens): ens = 0
    delta = ens - wang
    arrow = '↑' if delta > 0.01 else ('↓' if delta < -0.01 else '≈')
    print(f'{cls:<22} {wang:>7.4f} {v1:>7.4f} {v2:>7.4f} {ens:>9.4f} {delta:>+7.4f} {arrow}')

print('-'*68)
wang_macro = np.mean(list(WANG_AUC.values()))
print(f"{'AUC macro':<22} {wang_macro:>7.4f} {m_v1_test['auc_macro']:>7.4f} "      f"{m_v2_test['auc_macro']:>7.4f} {m_ens_test['auc_macro']:>9.4f} "      f"{m_ens_test['auc_macro']-wang_macro:>+7.4f}")
print(f"{'mAP':<22} {'N/A':>7} {m_v1_test['map']:>7.4f} {m_v2_test['map']:>7.4f} "      f"{m_ens_test['map']:>9.4f}")
print()
print(f'Peso final: w_v1={best_w:.1f}, w_v2={1-best_w:.1f}')
print(f'Mejora sobre v1:   {m_ens_test["auc_macro"]-m_v1_test["auc_macro"]:+.4f}')
print(f'Mejora sobre Wang: {m_ens_test["auc_macro"]-wang_macro:+.4f}')


## 7. Guardar resultados del ensemble

In [ ]:
import json

ensemble_results = {
    'weight_v1':       float(best_w),
    'weight_v2':       float(1 - best_w),
    'checkpoint_v1':   str(f'{PROJECT_ROOT}/experiments_s4ml/checkpoints/sprint4ml_phase2_best.pt'),
    'checkpoint_v2':   str(f'{PROJECT_ROOT}/experiments_s4ml_v2/checkpoints/phase2_best.pt'),
    'num_layers_v1':   4,
    'num_layers_v2':   6,
    'test_auc_macro':  float(m_ens_test['auc_macro']),
    'test_map':        float(m_ens_test['map']),
    'test_auc_per_class': {
        CLASS_NAMES_14[i]: float(m_ens_test['auc_per_class'][i])
        for i in range(14)
    },
    'wang_2017_auc_macro': float(wang_macro),
    'delta_vs_wang':   float(m_ens_test['auc_macro'] - wang_macro),
}

out_path = f'{PROJECT_ROOT}/experiments_s4ml_v2/ensemble_results.json'
with open(out_path, 'w') as f:
    json.dump(ensemble_results, f, indent=2, ensure_ascii=False)
print(f'✓ Guardado: {out_path}')

print()
print('RESUMEN FINAL DEL SPRINT 4:')
print(f'  Ensemble AUC macro (test): {m_ens_test["auc_macro"]:.4f}')
print(f'  Wang et al. 2017:          {wang_macro:.4f}')
print(f'  Delta:                     {m_ens_test["auc_macro"]-wang_macro:+.4f}')
print(f'  Clases sobre Wang:         {sum(1 for i in range(14) if not np.isnan(m_ens_test["auc_per_class"][i]) and m_ens_test["auc_per_class"][i] > list(WANG_AUC.values())[i])}/14')


## 8. Gráfica final

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(CLASS_NAMES_14))
w = 0.2

ax.bar(x-1.5*w, [WANG_AUC[c] for c in CLASS_NAMES_14],
       w, label='Wang 2017', color='#9E9E9E')
ax.bar(x-0.5*w, [m_v1_test['auc_per_class'][i] for i in range(14)],
       w, label=f'v1 ({m_v1_test["auc_macro"]:.3f})', color='#2196F3', alpha=0.8)
ax.bar(x+0.5*w, [m_v2_test['auc_per_class'][i] if not np.isnan(m_v2_test['auc_per_class'][i]) else 0 for i in range(14)],
       w, label=f'v2 ({m_v2_test["auc_macro"]:.3f})', color='#FF9800', alpha=0.8)
ax.bar(x+1.5*w, [m_ens_test['auc_per_class'][i] if not np.isnan(m_ens_test['auc_per_class'][i]) else 0 for i in range(14)],
       w, label=f'Ensemble ({m_ens_test["auc_macro"]:.3f})', color='#F44336', alpha=0.9)

ax.axhline(wang_macro, color='gray', linestyle='--', alpha=0.5,
           label=f'Wang macro={wang_macro:.3f}')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES_14, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('AUC'); ax.set_ylim(0.5, 1.0)
ax.legend(fontsize=9); ax.grid(alpha=0.3, axis='y')
ax.set_title(f'Sprint 4: Wang 2017 vs v1 vs v2 vs Ensemble (test set)')
plt.tight_layout()

os.makedirs(f'{PROJECT_ROOT}/experiments_s4ml_v2/figures', exist_ok=True)
plt.savefig(f'{PROJECT_ROOT}/experiments_s4ml_v2/figures/ensemble_final.png',
            dpi=120, bbox_inches='tight')
plt.show()
print('✓ Gráfica guardada.')
